# 🚀 Inspector AI — Treinamento no Google Colab (GPU)

Este notebook treina o Inspector AI com **60.000+ pares** usando a GPU gratuita do Google Colab.

## O que este notebook faz:
1. Instala dependências (PyTorch com CUDA, datasets, tokenizers, customtkinter)
2. Faz upload dos arquivos do projeto
3. Organiza os arquivos na estrutura esperada pelo script
4. Verifica a GPU disponível
5. Treina o modelo Transformer Seq2Seq
6. Baixa os arquivos treinados (.pth e tokenizer_vocab.json)

---
⚠️ **Importante:** Altere o runtime para **GPU** antes de executar:
`Runtime` → `Change runtime type` → `GPU`

📁 **Arquivos para fazer upload:**
- `scripts/train_colab.py`
- `scripts/model.py`
- `scripts/tokenizer.py`
- `inspector/data/processed/expanded_dataset.json` (ou `openassistant_dataset.json` ou `mock_dataset.json`)

💡 **Dica:** Se não tiver certeza de qual dataset usar, faça upload de todos os três `.json`. O script escolhe automaticamente o melhor disponível.

In [ ]:
# ============================================================================
# 1. INSTALAR DEPENDÊNCIAS
# ============================================================================
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q datasets tokenizers customtkinter
print("✅ Dependências instaladas")

In [ ]:
# ============================================================================
# 2. VERIFICAR GPU
# ============================================================================
import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Dispositivo:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("⚠️  Nenhuma GPU detectada. Altere o runtime para GPU.")

In [ ]:
# ============================================================================
# 3. FAZER UPLOAD DOS ARQUIVOS
# ============================================================================
from google.colab import files

print("Selecione os seguintes arquivos para upload:")
print("  1. scripts/train_colab.py")
print("  2. scripts/model.py")
print("  3. scripts/tokenizer.py")
print("  4. inspector/data/processed/expanded_60k_dataset.json  (60k pares - RECOMENDADO)")
print("     OU inspector/data/processed/expanded_dataset.json (20k pares)")
print("     OU inspector/data/processed/openassistant_dataset.json (9.2k pares)")
print("     OU inspector/data/processed/mock_dataset.json (600 pares)")
print("
Aguarde o seletor de arquivos...")
uploaded = files.upload()

print(f"\n✅ {len(uploaded)} arquivo(s) enviado(s):")
for name in uploaded:
    print(f"   - {name}")

In [ ]:
# ============================================================================
# 4. ORGANIZAR ARQUIVOS NA ESTRUTURA ESPERADA
# ============================================================================
import shutil
from pathlib import Path

content = Path('/content')

# Criar estrutura de pastas
(content / 'inspector' / 'data' / 'processed').mkdir(parents=True, exist_ok=True)
(content / 'inspector' / 'models').mkdir(parents=True, exist_ok=True)
(content / 'scripts').mkdir(parents=True, exist_ok=True)

# Mover arquivos para os lugares corretos
files_to_move = {
    'train_colab.py': content / 'scripts' / 'train_colab.py',
    'model.py': content / 'scripts' / 'model.py',
    'tokenizer.py': content / 'scripts' / 'tokenizer.py',
    'expanded_dataset.json': content / 'inspector' / 'data' / 'processed' / 'expanded_dataset.json',
    'openassistant_dataset.json': content / 'inspector' / 'data' / 'processed' / 'openassistant_dataset.json',
    'mock_dataset.json': content / 'inspector' / 'data' / 'processed' / 'mock_dataset.json',
}

for src_name, dest_path in files_to_move.items():
    src = content / src_name
    if src.exists():
        shutil.move(str(src), str(dest_path))
        print(f"✅ Movido: {src_name} → {dest_path}")
    else:
        print(f"⚠️  Não encontrado: {src_name}")

# Verificar o que está disponível
print("\n📁 Arquivos disponíveis:")
for p in sorted(content.rglob('*')):
    if p.is_file():
        print(f"   {p.relative_to(content)}")

In [ ]:
# ============================================================================
# 4. CONFIGURAR CAMINHOS E IMPORTS
# ============================================================================
import sys

sys.path.insert(0, '/content')
sys.path.insert(0, '/content/scripts')

from tokenizer import BPETokenizer, SOS_IDX, EOS_IDX, PAD_IDX, SPECIAL_TOKENS
from model import InspectorTransformer
from train_colab import pick_device, DEVICE, VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, D_FF, MAX_LEN, BATCH_SIZE, EPOCHS, LR, MAX_PAIRS, PAD_IDX

print(f"🖥️  Device selecionado: {DEVICE}")
print(f"📊 Hiperparâmetros:")
print(f"   Vocabulário: {VOCAB_SIZE}")
print(f"   D_MODEL: {D_MODEL}")
print(f"   Num Heads: {NUM_HEADS}")
print(f"   Num Layers: {NUM_LAYERS}")
print(f"   D_FF: {D_FF}")
print(f"   Max Len: {MAX_LEN}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LR}")

In [ ]:
# ============================================================================
# 5. EXECUTAR TREINAMENTO
# ============================================================================
from train_colab import main
main()

In [ ]:
# ============================================================================
# 6. BAIXAR ARQUIVOS TREINADOS
# ============================================================================
from google.colab import files

model_path = '/content/inspector/models/inspector_model.pth'
tokenizer_path = '/content/inspector/models/tokenizer_vocab.json'

import os

if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"📦 Baixando modelo: {model_path} ({size_mb:.2f} MB)")
    files.download(model_path)
else:
    print("⚠️  Modelo não encontrado. O treinamento pode não ter concluído ou falhou.")

if os.path.exists(tokenizer_path):
    print(f"🔤 Baixando tokenizer: {tokenizer_path}")
    files.download(tokenizer_path)
else:
    print("⚠️  Tokenizer não encontrado.")

print("\n✅ Pronto! Copie os arquivos para InspectorAI/inspector/models/ localmente.")